# 01 - Runtime smoke test setup

This notebook checks the BUSI VGG16 runtime without fitting, fine-tuning, or updating weights. It checks package imports, dataset discovery and a small real-image batch, builds the frozen ImageNet VGG16 baseline, runs inference, saves/reloads a temporary model, and optionally inspects BUS-BRA metadata.

**No dataset is downloaded or extracted. No model is trained.** The requirements installation and pretrained ImageNet weight download remain opt-in.

## Colab paths configured by this notebook

- `PROJECT_ROOT`: `/content/drive/MyDrive/BreastCancerProject/Breast-Cancer-VGG16-Colab-Smoke-Test`
- `BUSI_ROOT`: `/content/busi_dataset/BUSI` (extract BUSI yourself before running; this notebook does not extract the supplied ZIP)
- BUSI ZIP reference: `/content/drive/MyDrive/BreastCancerProject/BUSI.zip`
- BUS-BRA ZIP: `/content/drive/MyDrive/BreastCancerProject/BUSBRA.zip`
- `OUTPUT_ROOT`: `/content/drive/MyDrive/BreastCancerProject/runtime_smoke_test`

Run cells in order. If Drive is not mounted, run the optional mount cell after setting its flag to `True`, then continue. If BUSI folders are missing, the data checks will stop and explain the required location. No synthetic data is substituted. The BUSI binary baseline excludes `normal_scan` and mask files; expected counts are 437 benign and 210 malignant images.


# 02 — Detect Google Colab

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_MOUNTED = bool(IN_COLAB and DRIVE_ROOT.is_dir())
print("Running in Google Colab:", IN_COLAB)
print("Google Drive mounted:", DRIVE_MOUNTED)
if IN_COLAB and not DRIVE_MOUNTED:
    print("Drive is not mounted. To access the configured project and ZIP files, set USE_GOOGLE_DRIVE = True in the optional Drive mount cell and run it before continuing.")
elif not IN_COLAB:
    print("Colab Drive mounting is unavailable; use local environment variables or local default paths.")


# 03 - Optional Google Drive mount

This cell mounts Drive only when you change `USE_GOOGLE_DRIVE` to `True` and explicitly run the cell. It does not upload files, extract archives, or download datasets. If Drive is already mounted, you can leave the flag `False`.


In [ ]:
USE_GOOGLE_DRIVE = False  # Change to True and explicitly run only if Drive is not already mounted.
if USE_GOOGLE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Google Drive mounting is available only in Google Colab.")
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = Path("/content/drive/MyDrive").is_dir()
    if not DRIVE_MOUNTED:
        raise FileNotFoundError("Google Drive mount did not expose /content/drive/MyDrive. Complete the Colab authorization and rerun this cell.")
    print("Google Drive mounted and available.")
elif IN_COLAB and not Path("/content/drive/MyDrive").is_dir():
    print("Drive mount skipped. Set USE_GOOGLE_DRIVE = True and rerun this cell, then continue to path configuration.")
else:
    print("Drive mount skipped; Drive is already mounted or this is not Colab.")


# 04 - Configure project, dataset, archive, and output paths


In [ ]:
import os
import tempfile
from pathlib import Path

if IN_COLAB:
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive is not mounted. Run the optional Drive mount cell (set USE_GOOGLE_DRIVE = True), then rerun this path-configuration cell.")
    _DEFAULT_PROJECT_ROOT = Path("/content/drive/MyDrive/BreastCancerProject/Breast-Cancer-VGG16-Colab-Smoke-Test")
    _DEFAULT_DATA_ROOT = Path("/content/busi_dataset/BUSI")
    _DEFAULT_BUSI_ZIP = Path("/content/drive/MyDrive/BreastCancerProject/BUSI.zip")
    _DEFAULT_BUSBRA = Path("/content/drive/MyDrive/BreastCancerProject/BUSBRA.zip")
    _DEFAULT_OUTPUT_ROOT = Path("/content/drive/MyDrive/BreastCancerProject/runtime_smoke_test")
else:
    _DEFAULT_PROJECT_ROOT = Path.cwd()
    _DEFAULT_DATA_ROOT = _DEFAULT_PROJECT_ROOT / "data" / "BUSI"
    _DEFAULT_BUSI_ZIP = _DEFAULT_PROJECT_ROOT / "data" / "BUSI.zip"
    _DEFAULT_BUSBRA = _DEFAULT_PROJECT_ROOT / "data" / "BUSBRA.zip"
    _DEFAULT_OUTPUT_ROOT = Path(tempfile.gettempdir()) / "busi_smoke_test"

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", str(_DEFAULT_PROJECT_ROOT))).expanduser().resolve()
BUSI_ROOT = Path(os.environ.get("BUSI_ROOT", str(_DEFAULT_DATA_ROOT))).expanduser().resolve()
BUSI_ZIP = Path(os.environ.get("BUSI_ZIP", str(_DEFAULT_BUSI_ZIP))).expanduser().resolve()
BUSBRA_ZIP = Path(os.environ.get("BUSBRA_ZIP", str(_DEFAULT_BUSBRA))).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", str(_DEFAULT_OUTPUT_ROOT))).expanduser().resolve()

if not (PROJECT_ROOT / "src" / "model.py").is_file():
    raise FileNotFoundError(f"Project source was not found under {PROJECT_ROOT}. Confirm the project folder is available at the configured path or set PROJECT_ROOT to the checkout containing src/model.py and requirements.txt.")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BUSI_ROOT:", BUSI_ROOT)
print("BUSI ZIP (reference only; never extracted automatically):", BUSI_ZIP, "exists:", BUSI_ZIP.is_file())
print("BUSBRA_ZIP (optional):", BUSBRA_ZIP, "exists:", BUSBRA_ZIP.is_file())
print("OUTPUT_ROOT (temporary smoke artifacts):", OUTPUT_ROOT)


# 05 — Optional requirements installation

Dependencies are **not** installed automatically. To install them in this notebook runtime, change `RUN_REQUIREMENTS_INSTALL` to `True` and explicitly run this cell. This does not install anything in your local environment unless you run the cell there.

In [ ]:
import subprocess
import sys

RUN_REQUIREMENTS_INSTALL = False
if RUN_REQUIREMENTS_INSTALL:
    requirements_file = PROJECT_ROOT / "requirements.txt"
    if not requirements_file.is_file():
        raise FileNotFoundError(f"requirements.txt not found: {requirements_file}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)]
    )
    print("Requirements installation completed. Restart the runtime if TensorFlow was updated.")
else:
    print("Installation skipped. Install requirements only if you explicitly choose to run this cell.")

# 06 — Import runtime dependencies and project modules

In [ ]:
import csv
import inspect
import re
import zipfile
from io import BytesIO

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import sklearn

import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.config import ProjectConfig
from src.model import build_vgg16_model
from src.preprocessing import (
    CLASS_NAMES,
    discover_busi_images,
    make_tf_dataset,
)

print("Imports completed.")

# 07 — Runtime versions and device availability

GPU detection is informational. The smoke inference is forced onto CPU where TensorFlow permits it.

In [ ]:
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("scikit-learn:", sklearn.__version__)
print("Pandas:", pd.__version__)
GPU_DEVICES = tf.config.list_physical_devices("GPU")
CPU_DEVICES = tf.config.list_physical_devices("CPU")
print("Detected GPUs:", GPU_DEVICES)
print("Detected CPUs:", CPU_DEVICES)
try:
    tf.config.set_visible_devices([], "GPU")
    print("GPU hidden for this smoke test; model and inference will use CPU.")
except RuntimeError as exc:
    print("TensorFlow devices were already initialized; later smoke operations use /CPU:0 context where possible.")
    print("Device configuration detail:", exc)

# 08 — Discover BUSI images and verify expected classes

In [ ]:
BUSI_STATUS = "BLOCKED"
BUSI_RECORDS = []
BUSI_COUNTS = {name: 0 for name in CLASS_NAMES}
if not BUSI_ROOT.is_dir():
    print("BUSI check BLOCKED: directory not found:", BUSI_ROOT)
    print("Place the already-extracted BUSI dataset at BUSI_ROOT, or set BUSI_ROOT to its folder. The notebook will not download or extract BUSI_ZIP:", BUSI_ZIP)
else:
    required_busi_folders = ("benign_tumor", "malignant_tumor", "normal_scan")
    missing_busi_folders = [name for name in required_busi_folders if not (BUSI_ROOT / name).is_dir()]
    if missing_busi_folders:
        print("BUSI check BLOCKED: expected folders missing under", BUSI_ROOT, ":", missing_busi_folders)
        print("Expected the BUSI layout: benign_tumor/, malignant_tumor/, normal_scan/. No archive will be extracted automatically.")
    else:
        try:
            BUSI_RECORDS = discover_busi_images(BUSI_ROOT)
            BUSI_COUNTS = {
                "benign": sum(record.label == 0 for record in BUSI_RECORDS),
                "malignant": sum(record.label == 1 for record in BUSI_RECORDS),
            }
            normal_included = any(
                "normal_scan" in {part.lower() for part in record.path.parts}
                for record in BUSI_RECORDS
            )
            masks_included = any("_mask" in record.path.stem.lower() for record in BUSI_RECORDS)
            print("Class counts:", BUSI_COUNTS)
            print("Binary total:", len(BUSI_RECORDS))
            print("Normal images included:", normal_included)
            print("Mask files included:", masks_included)
            expected = BUSI_COUNTS == {"benign": 437, "malignant": 210}
            BUSI_STATUS = "PASS" if expected and len(BUSI_RECORDS) == 647 and not normal_included and not masks_included else "FAIL"
            print("BUSI discovery status:", BUSI_STATUS)
            if BUSI_STATUS != "PASS":
                print("Expected 437 benign + 210 malignant = 647. Verify BUSI release and BUSI_ROOT before continuing.")
        except Exception as exc:
            BUSI_STATUS = "FAIL"
            print("BUSI discovery failed:", type(exc).__name__, str(exc))


# 09 — Load a small real BUSI batch (no training)

In [ ]:
BATCH_STATUS = "BLOCKED"
BATCH_IMAGES = None
BATCH_LABELS = None
if BUSI_STATUS == "PASS":
    benign_records = [record for record in BUSI_RECORDS if record.label == 0]
    malignant_records = [record for record in BUSI_RECORDS if record.label == 1]
    sample_records = benign_records[:2] + malignant_records[:2]
    smoke_dataset = make_tf_dataset(
        sample_records,
        batch_size=4,
        image_height=224,
        image_width=224,
        training=False,
        seed=42,
        mode="resize",
    )
    with tf.device("/CPU:0"):
        BATCH_IMAGES, BATCH_LABELS = next(iter(smoke_dataset))
    BATCH_IMAGES = BATCH_IMAGES.numpy()
    BATCH_LABELS = BATCH_LABELS.numpy()
    BATCH_STATUS = "PASS"
    print("Loaded batch without training:", BATCH_IMAGES.shape)
else:
    print("Batch loading BLOCKED because the BUSI discovery check did not pass. No substitute images were created.")

# 10 — Verify batch shape, channel replication, scaling, and labels

In [ ]:
BATCH_VALIDATION_STATUS = "BLOCKED"
if BATCH_STATUS == "PASS":
    checks = {
        "shape": BATCH_IMAGES.ndim == 4 and BATCH_IMAGES.shape[1:] == (224, 224, 3),
        "three grayscale channels identical": (
            np.allclose(BATCH_IMAGES[..., 0], BATCH_IMAGES[..., 1])
            and np.allclose(BATCH_IMAGES[..., 1], BATCH_IMAGES[..., 2])
        ),
        "pixels in [0, 1]": float(BATCH_IMAGES.min()) >= 0.0 and float(BATCH_IMAGES.max()) <= 1.0,
        "two-class one-hot labels": (
            BATCH_LABELS.ndim == 2
            and BATCH_LABELS.shape[1] == 2
            and np.all(np.isin(BATCH_LABELS, [0.0, 1.0]))
            and np.allclose(BATCH_LABELS.sum(axis=1), 1.0)
        ),
    }
    for name, passed in checks.items():
        print(f"{'PASS' if passed else 'FAIL'}: {name}")
    BATCH_VALIDATION_STATUS = "PASS" if all(checks.values()) else "FAIL"
else:
    print("Batch validation BLOCKED because no verified real BUSI batch is available.")

# 11 — Construct the paper-faithful VGG16 baseline

The project builder requests ImageNet weights. To prevent an unexpected model-weight download, this cell requires the VGG16 no-top weights to be cached unless you explicitly set `ALLOW_IMAGENET_WEIGHT_DOWNLOAD = True`. This flag concerns pretrained model weights only; no dataset is downloaded.

In [ ]:
ALLOW_IMAGENET_WEIGHT_DOWNLOAD = False
KERAS_HOME = Path(os.environ.get("KERAS_HOME", str(Path.home() / ".keras"))).expanduser()
VGG16_WEIGHTS_CACHE = KERAS_HOME / "models" / "vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"
MODEL_STATUS = "BLOCKED"
SMOKE_MODEL = None
if not VGG16_WEIGHTS_CACHE.is_file() and not ALLOW_IMAGENET_WEIGHT_DOWNLOAD:
    print("Model construction BLOCKED: ImageNet VGG16 weights are not cached at:", VGG16_WEIGHTS_CACHE)
    print("If you explicitly permit fetching pretrained model weights, set ALLOW_IMAGENET_WEIGHT_DOWNLOAD = True and rerun this cell.")
else:
    try:
        with tf.device("/CPU:0"):
            SMOKE_MODEL = build_vgg16_model()
        MODEL_STATUS = "PASS"
        print("Baseline model constructed with the project's default ImageNet VGG16 configuration.")
    except Exception as exc:
        MODEL_STATUS = "FAIL"
        print("Model construction failed:", type(exc).__name__, str(exc))

# 12 — Print model summary and verify paper architecture

In [ ]:
ARCHITECTURE_STATUS = "BLOCKED"
if MODEL_STATUS == "PASS":
    SMOKE_MODEL.summary()
    backbone = SMOKE_MODEL.get_layer("vgg16")
    model_source = inspect.getsource(build_vgg16_model)
    checks = {
        "VGG16 include_top=False and ImageNet weights": (
            'include_top=False' in model_source and 'weights="imagenet"' in model_source
        ),
        "frozen convolutional base": (not backbone.trainable and all(not layer.trainable for layer in backbone.layers)),
        "VGG16 ends at block5_pool (no classifier top)": backbone.layers[-1].name == "block5_pool",
        "GlobalAveragePooling2D": isinstance(SMOKE_MODEL.get_layer("global_average_pooling"), tf.keras.layers.GlobalAveragePooling2D),
        "Dense(256, relu)": (
            SMOKE_MODEL.get_layer("dense_256").units == 256
            and SMOKE_MODEL.get_layer("dense_256").activation.__name__ == "relu"
        ),
        "Dropout(0.5)": np.isclose(SMOKE_MODEL.get_layer("dropout_0_5").rate, 0.5),
        "Dense(2, softmax)": (
            SMOKE_MODEL.get_layer("class_probabilities").units == 2
            and SMOKE_MODEL.get_layer("class_probabilities").activation.__name__ == "softmax"
        ),
    }
    for name, passed in checks.items():
        print(f"{'PASS' if passed else 'FAIL'}: {name}")
    ARCHITECTURE_STATUS = "PASS" if all(checks.values()) else "FAIL"
else:
    print("Architecture inspection BLOCKED because model construction did not pass.")

# 13 — One CPU forward pass (no gradients or weight updates)

In [ ]:
PREDICTION_STATUS = "BLOCKED"
PREDICTIONS_BEFORE_SAVE = None
if MODEL_STATUS == "PASS" and BATCH_VALIDATION_STATUS == "PASS":
    try:
        with tf.device("/CPU:0"):
            PREDICTIONS_BEFORE_SAVE = SMOKE_MODEL(BATCH_IMAGES[:1], training=False).numpy()
        valid_predictions = (
            PREDICTIONS_BEFORE_SAVE.shape == (1, 2)
            and np.all(np.isfinite(PREDICTIONS_BEFORE_SAVE))
            and np.allclose(PREDICTIONS_BEFORE_SAVE.sum(axis=1), 1.0, atol=1e-5)
        )
        PREDICTION_STATUS = "PASS" if valid_predictions else "FAIL"
        print("Prediction shape:", PREDICTIONS_BEFORE_SAVE.shape)
        print("Prediction:", PREDICTIONS_BEFORE_SAVE)
        print("Forward-pass status:", PREDICTION_STATUS)
    except Exception as exc:
        PREDICTION_STATUS = "FAIL"
        print("Forward pass failed:", type(exc).__name__, str(exc))
else:
    print("Forward pass BLOCKED; it requires the constructed model and verified real BUSI batch.")

# 14 — Save the untrained smoke model to a temporary directory

In [ ]:
SAVE_STATUS = "BLOCKED"
SMOKE_TEMP_DIRECTORY = None
SMOKE_MODEL_PATH = None
if MODEL_STATUS == "PASS":
    try:
        OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
        SMOKE_TEMP_DIRECTORY = tempfile.TemporaryDirectory(
            prefix="vgg16_runtime_smoke_", dir=str(OUTPUT_ROOT)
        )
        SMOKE_MODEL_PATH = Path(SMOKE_TEMP_DIRECTORY.name) / "untrained_smoke_model.keras"
        SMOKE_MODEL.save(SMOKE_MODEL_PATH)
        SAVE_STATUS = "PASS"
        print("Saved untrained smoke-test model temporarily:", SMOKE_MODEL_PATH)
    except Exception as exc:
        SAVE_STATUS = "FAIL"
        print("Temporary model save failed:", type(exc).__name__, str(exc))
else:
    print("Model save BLOCKED because model construction did not pass.")

# 15 — Reload the temporary model and verify inference

In [ ]:
RELOAD_STATUS = "BLOCKED"
RELOADED_MODEL = None
PREDICTIONS_AFTER_RELOAD = None
if SAVE_STATUS == "PASS":
    try:
        RELOADED_MODEL = tf.keras.models.load_model(SMOKE_MODEL_PATH)
        if BATCH_VALIDATION_STATUS == "PASS":
            with tf.device("/CPU:0"):
                PREDICTIONS_AFTER_RELOAD = RELOADED_MODEL(BATCH_IMAGES[:1], training=False).numpy()
            checks = (
                PREDICTIONS_AFTER_RELOAD.shape == (1, 2)
                and np.all(np.isfinite(PREDICTIONS_AFTER_RELOAD))
                and np.allclose(PREDICTIONS_AFTER_RELOAD.sum(axis=1), 1.0, atol=1e-5)
                and np.allclose(PREDICTIONS_BEFORE_SAVE, PREDICTIONS_AFTER_RELOAD, atol=1e-5)
            )
            RELOAD_STATUS = "PASS" if checks else "FAIL"
            print("Reloaded prediction shape:", PREDICTIONS_AFTER_RELOAD.shape)
            print("Saved/reloaded predictions match:", np.allclose(PREDICTIONS_BEFORE_SAVE, PREDICTIONS_AFTER_RELOAD, atol=1e-5))
        else:
            RELOAD_STATUS = "PARTIAL"
            print("Model reload succeeded; prediction comparison BLOCKED because no verified BUSI batch is available.")
    except Exception as exc:
        RELOAD_STATUS = "FAIL"
        print("Model reload/inference failed:", type(exc).__name__, str(exc))
else:
    print("Reload BLOCKED because a temporary model was not saved.")

# 16 — Inspect local BUS-BRA CSV, labels, and image paths (no training)

In [ ]:
BUSBRA_STATUS = "BLOCKED"
if not BUSBRA_ZIP.is_file():
    print("BUS-BRA inspection BLOCKED: archive not found:", BUSBRA_ZIP)
    print("Set BUSBRA_ZIP to an already available BUS-BRA.zip. No download will be attempted.")
else:
    try:
        with zipfile.ZipFile(BUSBRA_ZIP, "r") as archive:
            names = set(archive.namelist())
            csv_candidates = sorted(name for name in names if name.lower().endswith(".csv"))
            metadata_name = next(
                (name for name in csv_candidates if Path(name).name.lower() == "bus_data.csv"),
                None,
            )
            if metadata_name is None:
                raise FileNotFoundError("Archive has no bus_data.csv metadata file")
            metadata = pd.read_csv(BytesIO(archive.read(metadata_name)))
            if not {"ID", "Pathology"}.issubset(metadata.columns):
                raise ValueError("bus_data.csv must contain ID and Pathology columns")
            metadata["normalized_pathology"] = metadata["Pathology"].astype(str).str.strip().str.lower()
            labeled = metadata[metadata["normalized_pathology"].isin(["benign", "malignant"])].copy()
            image_paths = [f"BUSBRA/Images/{str(image_id).strip()}.png" for image_id in labeled["ID"]]
            missing_images = [path for path in image_paths if path not in names]
            label_counts = labeled["normalized_pathology"].value_counts().to_dict()
            print("Metadata file:", metadata_name)
            print("CSV rows:", len(metadata))
            print("Benign/malignant labeled rows:", len(labeled))
            print("Normalized pathology counts:", label_counts)
            print("Image paths found:", len(image_paths) - len(missing_images), "/", len(image_paths))
            if missing_images:
                print("Missing image examples:", missing_images[:5])
            BUSBRA_STATUS = "PASS" if not missing_images and len(labeled) > 0 else "FAIL"
            print("BUS-BRA metadata/path status:", BUSBRA_STATUS)
    except Exception as exc:
        BUSBRA_STATUS = "FAIL"
        print("BUS-BRA inspection failed:", type(exc).__name__, str(exc))

# 17 — Smoke-test summary and temporary cleanup

In [ ]:
SUMMARY = {
    "imports": "PASS" if "tf" in globals() and "build_vgg16_model" in globals() else "FAIL",
    "BUSI discovery/counts": BUSI_STATUS,
    "real batch and preprocessing checks": BATCH_VALIDATION_STATUS,
    "baseline model construction": MODEL_STATUS,
    "architecture verification": ARCHITECTURE_STATUS,
    "CPU forward pass": PREDICTION_STATUS,
    "temporary save/reload": RELOAD_STATUS,
    "BUS-BRA metadata/path inspection": BUSBRA_STATUS,
}
for name, status in SUMMARY.items():
    print(f"{status}: {name}")
if SMOKE_TEMP_DIRECTORY is not None:
    SMOKE_TEMP_DIRECTORY.cleanup()
    print("Temporary smoke model directory cleaned up.")
print("No training or fine-tuning was performed by this notebook.")